# 39 · ColBERT / SPLADE / 长上下文

> 稠密向量丢失词级信号，稀疏向量不懂语义。**ColBERT**（多向量）与 **SPLADE**（词级+学习）在补足这块。本课顺带聊**长上下文时代**对 RAG 的冲击。

**本文件覆盖知识点**：ColBERT / MaxSim / Late Interaction / SPLADE / 稀疏与稠密结合 / Long Context / Lost in the Middle

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. ColBERT：Late Interaction + MaxSim

```text
Bi-Encoder:  query→1个向量 × doc→1个向量 = 点积(粗，丢词)   
ColBERT:     query→N个词向量  doc→M个词向量
评分 = 每个 query 词向量 去 doc 里找最相似的词向量(MaxSim) 再求和
→ 保留了“哪个词对上哪个词”的精确度，又享受了向量检索的速度
```

- **Late Interaction**：交互放在最后打分阶段（不做全量交叉，仍可走 ANN 预筛）；
- **MaxSim**：query 每个 token 与 doc 全部 token 的最大相似度之和。

### 本课的 MaxSim 是用真模型「近似」出来的（重要局限）

ColBERT 的核心是**一个 token 级的多向量编码器**：它输出「每个 token 一个向量」，且这些向量
是**带上下文**的（同一个词在不同句子里向量不同）。而 `text-embedding-v3` 是「整段文本 → 一个
向量」的编码器，并不提供 token 级多向量输出。

所以下面代码的做法是：把**每个 token 单独**送进 `text-embedding-v3` 取向量——
- 向量是**真实**的（真调模型，1024 维），token 间相似度也是真实的；
- 但编码器**不是 ColBERT**：token 向量丢掉了上下文，也没有 ColBERT 的 query/doc 双塔结构；
- 因此它演示的是 **MaxSim 的评分机制与直觉**，不能当作 ColBERT 的效果复现，
  更不能用这些分数去对比 ColBERT 的论文指标。要真做 ColBERT，需要专门的 token 级编码器。

### ColBERT 的工程配合
- 先 ANN 召回粗候选 → 只对候选做 ColBERT 精排 → 精度/成本折中；
- 需存储“文档的多向量”，索引体积比普通稠密大，可用残差压缩。

In [ ]:
# 用逐 token 真调 embedding 跑一次 MaxSim —— 看 Late Interaction 的评分直觉
# ⚠ 局限：text-embedding-v3 是「整段文本 → 一个向量」的编码器，没有 ColBERT 那种 token 级
#    多向量输出。这里是把每个 token 单独送进模型拿向量：向量是真的，但编码器不是 ColBERT
#    （token 向量不带上下文），所以这是 MaxSim 机制的近似演示，不是 ColBERT 复现。
import numpy as np

q_tokens = ['苹果', '手机', '价格', '上涨', '趋势', '分析']
docs = {
    'D1 强相关': ['苹果', '手机', '价格', '上涨', '明显'],
    'D2 部分相关': ['华为', '手机', '降价', '促销', '活动'],
    'D3 无关': ['今天', '天气', '很好', '适合', '出游'],
}

uniq = sorted({t for t in q_tokens} | {t for ts in docs.values() for t in ts})
TV = embed(uniq)                                   # 真调 text-embedding-v3，一次批量编码全部 token
tv = {t: TV[i] for i, t in enumerate(uniq)}
print('token 向量矩阵：%d 个 token × %d 维（真调 %s，已 L2 归一化）'
      % (TV.shape[0], TV.shape[1], EMBED_MODEL))

def maxsim(qtoks, dtoks):
    """MaxSim：每个 query token 去 doc 全部 token 里找最相似的一个，再求和"""
    S = np.array([[float(tv[q] @ tv[d]) for d in dtoks] for q in qtoks])
    return float(S.max(axis=1).sum()), S

print()
scores = {}
for name, dtoks in docs.items():
    sc, S = maxsim(q_tokens, dtoks)
    scores[name] = sc
    print('%s  →  MaxSim 总分 %.3f' % (name, sc))
    print('    打分矩阵(行=query token, 列=doc token):')
    print('           ' + ' '.join('%7s' % d for d in dtoks))
    for i, qt in enumerate(q_tokens):
        print('    %-6s ' % qt + ' '.join('%7.3f' % v for v in S[i]))
    print('    每个 query token 的最佳匹配: ' + ' '.join('%.3f' % v for v in S.max(axis=1)))

# 对照：传统双塔（Bi-Encoder）——query 整段一个向量、doc 整段一个向量，直接点积
full = embed([' '.join(q_tokens)] + [' '.join(ts) for ts in docs.values()])
q_vec, d_vecs = full[0], full[1:]
print()
print('对照 · 整段向量直接点积（Bi-Encoder 的做法，也是本课「粗」的那一路）：')
for name, dv in zip(docs, d_vecs):
    print('    %s  点积 %.3f' % (name, float(q_vec @ dv)))

print()
best = max(scores, key=scores.get)
print('→ MaxSim 最高分：%s（%.3f）；无关文档 %s（%.3f），差距 %.3f。'
      % (best, scores[best], 'D3 无关', scores['D3 无关'], scores[best] - scores['D3 无关']))
print('  MaxSim 让每个查询词各自去找最像的文档词：D1 的「苹果/价格/上涨」都能对上同名词，'
      '而 D3 的词跟查询毫无交集、只能勉强凑分。整段点积只给一个总分，看不出「哪个词对上了」。')
print('  注意真实值的两个特征（随机向量演示里看不到）：① 真模型对短词的余弦有很高的「地板值」'
      '（连无关 token 也在 0.5 上下），所以 MaxSim 的绝对分差不夸张，判优劣要看相对排序；'
      '② 同词自匹配是 1.000，但近义词（苹果↔华为 0.858）才有区分度，这正是词级交互的价值。')
if not _HAS_KEY:
    print('  （未配置 Key：token 向量直接读此前真实调用的缓存向量，维度与相似度仍是真实的）')

## 2. SPLADE：学出来的稀疏向量

SPLADE 用模型把文本变成**稀疏的 term 加权向量**（含同义扩展词），兼得：
- BM25 的可解释/词级精确；
- 稠密向量的语义扩展（“小轿车”也能激活“汽车”）。

```text
输入: 汽车修理
输出稀疏向量: 汽车:1.2  轿车:0.8  维修:1.0  保养:0.6 …（其它全 0）
→ 可直接放进倒排索引，与 BM25 系出同门
```

### 怎么选
| 模型 | 适用 | 代价 |
|------|------|------|
| 稠密向量 | 语义相似为主 | 低（1 向量/文档） |
| **ColBERT** | 高精度精排/复杂匹配 | 多向量、存储高 |
| **SPLADE** | 需词级精确+语义、可解释 | 推理成本较高 |

In [ ]:
# 知识点·真调说明：检索模型选型 —— 三个业务场景让模型对号入座讲权衡
_llm_live(
    prompt='为下面三个检索场景各选一个主方案（稠密向量 / ColBERT / SPLADE / 混合），并用一句话说理由：\n'
           '① 法律合同：必须精确命中条款编号与原文措辞，查不到即错；\n'
           '② 电商“找同款”：买家描述五花八门，语义相近即可；\n'
           '③ 故障工单：问题口语化，又要兼顾“机型/报错码”等精确词。',
    system='你是检索系统架构师。请按“①选型：理由”逐条回答，直接给结论，不展开教材式背景。',
    fallback='未配置 Key 的固定样例：\n'
             '① 法律合同：SPLADE，或 SPLADE/BM25 保词级命中 + 稠密语义兜底的混合——条款编号只能靠词级命中。\n'
             '② 电商同款：稠密向量主 + ColBERT 精排——语义为主，多向量只在精排阶段补精确匹配。\n'
             '③ 故障工单：混合检索（稀疏命中型号/报错码 + 稠密接住口语化描述），存储与延迟允许再叠加 ColBERT 精排。',
    temperature=0.2,
)
print('→ 没有“最好的检索模型”：数据形态决定该保“词级”还是“语义”；工程上通常是几者组合而非单选。')

## 3. Long Context（长上下文）对 RAG 的影响

模型窗口越来越长（qwen-long 等达百万 token），带来新选择：

- 直接把文档塞进窗口的 **Long-Context 模式**：省检索、防漏信息；但贵、慢；
- **Lost in the Middle**：长上下文里模型对“中间位置”的信息记得最差——
  - 关键片段放**开头/结尾**；
  - 或仍用 RAG 把长文档压缩成最相关段落，再进窗口。

> 实践倾向：长上下文与 RAG **互补**——RAG 选段 + 长窗口兜底“还要更多上下文”时。



In [ ]:
# 知识点·真调说明：Lost-in-the-Middle —— 唯一答案放“开头 vs 正中间”，让长上下文找同一件事
_a = ['华鑫', '森屿', '远帆', '明澈', '嘉禾', '蓝湾', '北辰', '清晏', '卓然', '隽永', '瀚海', '青梧',
      '澄川', '曜石', '栖云', '赤松', '聆风', '沐光', '逸云', '观澜', '擎苍', '归鸿', '凝霜', '拂晓',
      '叠翠', '惊鸿', '鸣沙', '临渊', '竹影', '松风']
_suf = ['科技', '数据', '物联', '智造', '云服', '软件']
_svc = ['质检云', '设计库', '货运SaaS', '病历检索', '能源台账', '楼宇物联', '题库系统', '基因检索',
        '设备云', '文创库', '云盘', '农情监测', '水务台账', '安全监测', '点餐系统', '坯布管理',
        '行程助手', '健康档案', '车服百科', '保单库', '产线MES', '进销存', '代码检索', '咨询知识库',
        '苗木台账', '媒资库', '运单系统', '安防平台', '电子书库', '生产看板']

def _t(i):
    m = 11 + i // 28          # 分布在 2026-11 ~ 2027-05 之间
    y = 2026
    if m > 12:
        m -= 12
        y = 2027
    d = i % 28 + 1
    if (y, m, d) == (2027, 3, 18):   # 别把唯一目标日期 2027-03-18 也生成进干扰项
        d = 19
    return '%d-%02d-%02d' % (y, m, d)

_other = [(_a[i % 30] + _suf[i // 30], _svc[i % 30], _t(i)) for i in range(180)]
_target = ('启明智造', '设备巡检', '2027-03-18')

def _records(target_first):
    if target_first:
        items = [_target] + _other
    else:
        k = len(_other) // 2
        items = _other[:k] + [_target] + _other[k:]
    return '\n'.join('客户「%s」签约了「%s」，合同续约时间：%s。' % t for t in items)

_q = '请只输出一个日期（YYYY-MM-DD）：客户「启明智造」的合同续约时间是？'
print('① 唯一答案放最开头（共 181 条相似记录，约 5k token）')
head = _llm_live(
    prompt=_records(True) + '\n\n' + _q,
    system='你是信息提取助手，只依据给定资料回答，不要输出日期以外的任何文字。',
    fallback='未配置 Key 的固定样例：\n2027-03-18（答案在开头，轻松命中）',
    temperature=0.1,
)
print()
print('② 唯一答案埋在 181 条相似记录的正中间')
mid = _llm_live(
    prompt=_records(False) + '\n\n' + _q,
    system='你是信息提取助手，只依据给定资料回答，不要输出日期以外的任何文字。',
    fallback='未配置 Key 的固定样例：\n2027-01-16（长上下文里中间位置的唯一答案被干扰项淹没——'
             '同一句话放到开头就不会错，这就是 Lost-in-the-Middle）',
    temperature=0.1,
)
print()
def _hit(s):
    return s is not None and '2027-03-18' in s

if not _HAS_KEY:
    print('（无 Key 演示样例效果：开头版答对、中间版答错 → Lost-in-the-Middle）')
elif _hit(head) and _hit(mid):
    print('本次两处都答对：模型在更长上下文里依然把中间信息找了出来。'
          'Lost-in-the-Middle 是概率性现象：越长、噪声越多、越靠中间越容易丢，不是必然翻车。')
elif _hit(head) and not _hit(mid):
    print('→ 现场复现 Lost-in-the-Middle：同一答案放开头能答对，埋进正中间就被干扰带偏——位置确实影响命中。')
else:
    print('本次结果有波动（可重跑观察）。结论不变：上下文越长，越靠中间的信息越容易被模型忽略。')
print('→ 工程启示：窗口再长也不要“一股脑全塞”——先用 RAG 挑出最相关片段、把关键句放靠前位置，再进生成。')

## 小结

- **ColBERT** 用 MaxSim 晚交互保留词级精确度；
- **SPLADE** 用学习型稀疏向量兼得语义与词级；
- **长上下文**不是 RAG 的终点，注意 Lost-in-the-Middle，两者结合更优。